# Ćwiczenie 1 — IPython (dodatek) — 01.05 ciekawostki + 01.07 Timing/Profiling (`%prun`, `%lprun`)

Co tu dokładamy:
- **01.05 (bonus)**: `SList` (`grep`, `fields`, `.s`, `.n`), `%cd` vs `!cd`, `automagic`, `%env`
- **01.07**: `%time`, `%timeit`, `%prun` + ćwiczenia optymalizacji
- **01.07 (bonus/ważne)**: `%lprun` (line‑by‑line profiling) na przykładzie funkcji straty z ML

Założenia: **Linux / WSL / Colab** (na Windows prosimy o WSL albo Colab).

Styl taki sam: TODO + testy `assert` + ✅.


## Jak pracować
1) Uzupełnij TODO.
2) Odpal komórkę z testami — jeśli widzisz ✅, jest OK.
3) Jeśli magiki nie działają, upewnij się, że uruchamiasz w Jupyter/IPython.


In [1]:
import os
import math
import random
import numpy as np
import pstats
from IPython import get_ipython

RANDOM_SEED = 0
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

ip = get_ipython()
assert ip is not None, 'Ten notebook wymaga IPython/Jupyter (ipykernel).'
print('✅ Setup OK')


✅ Setup OK
The history saving thread hit an unexpected error (OperationalError('database is locked')).History will not be written to the database.


## 01.05 — dodatki: `SList`, `grep/fields`, `%cd` vs `!cd`, `automagic`, `%env`
W IPython komendy shellowe (`!ls`, `!pwd`, itd.) zwracają często specjalny typ listy: **`SList`**.
To wygląda jak zwykła lista stringów, ale ma metody typu `grep`, `fields` oraz właściwości `.s`, `.n`.


In [2]:
# Tworzymy sandbox (katalog + kilka plików), żeby ćwiczyć bez ryzyka.
SANDBOX_DIR = os.path.join(os.getcwd(), '_ipython_shell_sandbox')
os.makedirs(SANDBOX_DIR, exist_ok=True)

for name in ['data_01.csv', 'data_02.csv', 'notes.txt', 'readme.md']:
    path = os.path.join(SANDBOX_DIR, name)
    if not os.path.exists(path):
        with open(path, 'w', encoding='utf-8') as f:
            f.write(f'file={name}\n')

directory = !ls {SANDBOX_DIR}
directory


['data_01.csv', 'data_02.csv', 'notes.txt', 'readme.md']

### Ćwiczenie 5.A — `SList`: rozpoznanie + `grep`
1) Napisz `is_slist(obj)`.
2) `DATA_FILES` = pliki zaczynające się od `data_`.
3) `TXT_FILES` = pliki kończące się na `.txt`.


In [4]:
def is_slist(obj) -> bool:
    """Zwróć True, jeśli obj jest IPython.utils.text.SList (lub podtypem)."""
    # TODO isinstance   - obj / SList
    raise NotImplementedError

DATA_FILES = None  # TODO np. directory.grep(r'^data_')
TXT_FILES = None   # TODO


In [3]:
assert is_slist(directory) is True
assert hasattr(directory, 'grep') and hasattr(directory, 'fields')
assert set(DATA_FILES) == {'data_01.csv', 'data_02.csv'}
assert set(TXT_FILES) == {'notes.txt'}
print('✅ 5.A OK')


NameError: name 'is_slist' is not defined

### Ćwiczenie 5.B — `SList.fields()` na `ls -l`
Zrób listing `ls -l` i wyciągnij z niego nazwy plików.

Hint: `fields(8)` często daje nazwę pliku (kolumna z nazwą), a przy okazji pomija linię typu `total ...`.


In [5]:
long_listing = !ls -l {SANDBOX_DIR}
long_listing[:5]


['total 16',
 '-rw-rw-r-- 1 piotr piotr 17 lut 28 21:09 data_01.csv',
 '-rw-rw-r-- 1 piotr piotr 17 lut 28 21:09 data_02.csv',
 '-rw-rw-r-- 1 piotr piotr 15 lut 28 21:09 notes.txt',
 '-rw-rw-r-- 1 piotr piotr 15 lut 28 21:09 readme.md']

In [ ]:
FILENAMES_FROM_LS_L = None  # TODO: użyj long_listing.fields(...) i indeks - sprawdz, który 


In [ ]:
assert isinstance(FILENAMES_FROM_LS_L, list)
assert set(FILENAMES_FROM_LS_L) >= {'data_01.csv', 'data_02.csv', 'notes.txt', 'readme.md'}
print('✅ 5.B OK')


### Ćwiczenie 5.C — `%cd` vs `!cd`  (tylko wykonać i zobaczyć co cd robi lub raczej nie robi :) - zadanie zrobione )
Pokaż pułapkę: `!cd` nie zmienia katalogu kernela.
Następnie użyj `%cd` (albo `ip.run_line_magic('cd', ...)`) żeby wejść do sandboxu i wrócić.


In [ ]:
ORIG_CWD = os.getcwd()

start_pwd = !pwd[0]

!cd ..
after_shell_cd = !pwd[0]

ip.run_line_magic('cd', SANDBOX_DIR)
cwd_in_sandbox = os.getcwd()

ip.run_line_magic('cd', ORIG_CWD)
cwd_back = os.getcwd()



In [ ]:
assert after_shell_cd == start_pwd
assert os.path.abspath(cwd_in_sandbox) == os.path.abspath(SANDBOX_DIR)
assert os.path.abspath(cwd_back) == os.path.abspath(ORIG_CWD)
print('✅ 5.C OK')


### Ćwiczenie 5.D — `automagic` i `%env`
- Napisz 2–4 zdania: co robi `automagic` i jaki ma minus.
- Ustaw `COURSE_TAG=IPYTHON_01` przez `%env` (lub `ip.run_line_magic('env', ...)`) i odczytaj w Pythonie.


In [7]:
ANSWER_AUTOMAGIC = (
    "Automagic to tryb IPython, w którym część komend magic działa bez znaku %. "
    "Ułatwia pracę (jak w shellu), ale może powodować konflikty, gdy nazwy zmiennych/funkcji "
    "pokrywają się z nazwami komend (niejednoznaczność)."
)

# TODO: ustaw COURSE_TAG=IPYTHON_01 przez %env / ip.run_line_magic
COURSE_TAG = None


In [ ]:
assert isinstance(ANSWER_AUTOMAGIC, str) and len(ANSWER_AUTOMAGIC.strip()) >= 80
assert COURSE_TAG == 'IPYTHON_01'
print('✅ 5.D OK')


## 01.07 — Timing & Profiling: `%time`, `%timeit`, `%prun`, `%lprun` ( Zrób 7.A, reszta dla chętnych.)

###  `%prun` i `%lprun` — do czego służą?

- **`%prun`**  
  Profiluje całe wywołanie funkcji i pokazuje, **które funkcje zajmują najwięcej czasu** (podsumowanie na poziomie funkcji).  
  Przydatne do szybkiego znalezienia „wąskiego gardła” w programie.

- **`%lprun`** *(line_profiler, opcjonalne)*  
  Profiluje kod **linia po linii**, pokazując ile czasu zajmuje każda instrukcja.  
  Przydatne, gdy wiesz już, która funkcja jest wolna i chcesz zobaczyć **dokładnie gdzie w środku tracisz czas**.

 W skrócie:
- `%prun` → *która funkcja jest wolna*  
- `%lprun` → *która linia w funkcji jest wolna*

### Zrób 7.A, reszta dla chętnych.

Nawyk:
1) zmierz (`%timeit`)
2) zlokalizuj bottleneck (`%prun` / `%lprun`)
3) popraw (algorytm/wektoryzacja)
4) zmierz ponownie


### Ćwiczenie 7.A — `%timeit`: pętla vs NumPy
Zaimplementuj wersję wektorową i porównaj czasy.


In [ ]:
def sum_squares_loop(x: np.ndarray) -> float:
    total = 0.0
    for v in x:
        total += float(v) * float(v)
    return total

x = np.random.rand(200_000)
sum_squares_loop(x)


In [ ]:
def sum_squares_numpy(x: np.ndarray) -> float:
    # TODO: bez pętli
    raise NotImplementedError


In [ ]:
assert abs(sum_squares_loop(x) - sum_squares_numpy(x)) < 1e-6
print('✅ 7.A poprawność OK')


In [ ]:
t_loop = %timeit -o -n 3 -r 3 sum_squares_loop(x)
t_np   = %timeit -o -n 3 -r 3 sum_squares_numpy(x)
print('loop avg:', t_loop.average)
print('np   avg:', t_np.average)
print('speedup:', t_loop.average / t_np.average)


In [ ]:
assert hasattr(t_loop, 'average') and hasattr(t_np, 'average')
print('✅ 7.A timeit OK')


### Ćwiczenie 7.B — `%prun`: znajdź bottleneck (`sum_of_lists`)
Uruchom `%prun -r -s tottime ...` i zapisz wynik do zmiennej `stats_sum_lists`.


In [ ]:
def sum_of_lists(N: int) -> int:
    total = 0
    for i in range(5):
        L = [j ^ (j >> i) for j in range(N)]
        total += sum(L)
    return total

sum_of_lists(10_000)


In [ ]:
N_PROFILE = 20_000
stats_sum_lists = %prun -r -s tottime sum_of_lists(N_PROFILE)


In [ ]:
assert isinstance(stats_sum_lists, pstats.Stats)
print('✅ 7.B prun OK')


Teraz napisz funkcję, która z obiektu `pstats.Stats` wyciągnie nazwę funkcji o największym `tottime` spośród funkcji z tego notebooka (pomijamy wrappery typu `exec`, `<module>`).


In [ ]:
def top_user_function_by_tottime(stats: pstats.Stats) -> str:
    # TODO
    raise NotImplementedError

HOT_FUNCTION_SUM_LISTS = None  # TODO
ANSWER_SUM_LISTS_HOTSPOT = ""  # TODO: min. 120 znaków


In [ ]:
assert isinstance(HOT_FUNCTION_SUM_LISTS, str)
assert len(ANSWER_SUM_LISTS_HOTSPOT.strip()) >= 120
print('✅ 7.B hotspot OK')


### Ćwiczenie 7.C — optymalizacja `sum_of_lists` (generator + NumPy)
Zaimplementuj dwie wersje i porównaj czasy.


In [ ]:
def sum_of_lists_gen(N: int) -> int:
    # TODO: generator zamiast listy
    raise NotImplementedError

def sum_of_lists_numpy(N: int) -> int:
    # TODO: NumPy (np.arange + operacje bitowe)
    raise NotImplementedError


In [ ]:
ref = sum_of_lists(5_000)
assert sum_of_lists_gen(5_000) == ref
assert sum_of_lists_numpy(5_000) == ref
print('✅ 7.C poprawność OK')


In [ ]:
N_TIME = 200_000
t_base = %timeit -o -n 1 -r 3 sum_of_lists(N_TIME)
t_gen  = %timeit -o -n 1 -r 3 sum_of_lists_gen(N_TIME)
t_np   = %timeit -o -n 1 -r 3 sum_of_lists_numpy(N_TIME)

SPEEDUP_NUMPY = t_base.average / t_np.average
print('speedup numpy:', SPEEDUP_NUMPY)


In [ ]:
assert SPEEDUP_NUMPY > 2.0
print('✅ 7.C speedup OK')


### Ćwiczenie 7.D — algorytm “ML‑owy”: macierz odległości (pairwise distances)
Uzupełnij wersję szybką i porównaj do wolnej.


In [ ]:
def pairwise_sqeuclidean_loop(X: np.ndarray) -> np.ndarray:
    n = X.shape[0]
    D = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(n):
            diff = X[i] - X[j]
            D[i, j] = float(np.dot(diff, diff))
    return D

X_small = np.random.randn(20, 5)
D_small = pairwise_sqeuclidean_loop(X_small)


In [ ]:
def pairwise_sqeuclidean_fast(X: np.ndarray) -> np.ndarray:
    # TODO: bez pętli w Pythonie
    # Hint: ||a-b||^2 = ||a||^2 + ||b||^2 - 2 a·b
    raise NotImplementedError


In [ ]:
D_fast = pairwise_sqeuclidean_fast(X_small)
assert np.allclose(D_fast, D_small, atol=1e-8)
print('✅ 7.D poprawność OK')


In [ ]:
X_profile = np.random.randn(120, 20)
stats_pairwise = %prun -r -s tottime pairwise_sqeuclidean_loop(X_profile)
assert isinstance(stats_pairwise, pstats.Stats)
print('✅ 7.D prun OK')


In [ ]:
X_time = np.random.randn(200, 25)
t_loop_pw = %timeit -o -n 1 -r 3 pairwise_sqeuclidean_loop(X_time)
t_fast_pw = %timeit -o -n 3 -r 3 pairwise_sqeuclidean_fast(X_time)
print('speedup:', t_loop_pw.average / t_fast_pw.average)


### Ćwiczenie 7.E — `%lprun` (line profiler) i optymalizacja: logistic loss
To ćwiczenie jest celowo **ML‑owe**: liczymy średnią stratę logistyczną.

Kroki:
1) Załaduj `line_profiler` i uruchom `%lprun`.
2) Na podstawie wyniku wskaż w 1–3 zdaniach, które linie są bottleneck.
3) Zaimplementuj wersję wektorową i zmierz speedup.

Jeśli nie masz `line_profiler`:
- w Colab: `!pip -q install line_profiler`
- w WSL: `pip install line_profiler`
Potem zwykle **restart kernel**.


In [ ]:
HAS_LINE_PROFILER = False
try:
    ip.run_line_magic('load_ext', 'line_profiler')
    HAS_LINE_PROFILER = True
    print('✅ line_profiler loaded (%lprun działa)')
except Exception as e:
    print('❌ line_profiler nie jest dostępny. Zainstaluj: pip install line_profiler')
    print('   error:', e)


In [ ]:
def logistic_loss_loop(X: np.ndarray, y: np.ndarray, w: np.ndarray) -> float:
    loss = 0.0
    for i in range(len(y)):
        z = float(np.dot(X[i], w))
        loss += math.log(1.0 + math.exp(-float(y[i]) * z))
    return loss / float(len(y))

rng = np.random.default_rng(RANDOM_SEED)
X_lp = rng.normal(size=(20_000, 50))
w_lp = rng.normal(size=(50,))
y_lp = rng.choice([-1.0, 1.0], size=(20_000,))

logistic_loss_loop(X_lp[:1000], y_lp[:1000], w_lp)


In [ ]:
# TODO: jeśli HAS_LINE_PROFILER=True, uruchom:
# %lprun -f logistic_loss_loop logistic_loss_loop(X_lp, y_lp, w_lp)
# i zobacz które linie dominują.
pass


In [ ]:
ANSWER_LPRUN_BOTTLENECK = ""  # TODO: min. 60 znaków

def logistic_loss_vec(X: np.ndarray, y: np.ndarray, w: np.ndarray) -> float:
    # TODO: bez pętli w Pythonie
    # Hint: z = X @ w; użyj np.logaddexp(0, -y*z)
    raise NotImplementedError


In [ ]:
ref_lp = logistic_loss_loop(X_lp[:2000], y_lp[:2000], w_lp)
fast_lp = logistic_loss_vec(X_lp[:2000], y_lp[:2000], w_lp)
assert abs(ref_lp - fast_lp) < 1e-6
assert isinstance(ANSWER_LPRUN_BOTTLENECK, str) and len(ANSWER_LPRUN_BOTTLENECK.strip()) >= 60
print('✅ 7.E poprawność OK')


In [ ]:
t_lp_loop = %timeit -o -n 1 -r 3 logistic_loss_loop(X_lp, y_lp, w_lp)
t_lp_vec  = %timeit -o -n 3 -r 3 logistic_loss_vec(X_lp, y_lp, w_lp)
speedup_lp = t_lp_loop.average / t_lp_vec.average
print('speedup:', speedup_lp)


In [ ]:
assert speedup_lp > 5.0
print('✅ 7.E speedup OK')


##  Bonus (opcjonalnie): memory_profiler
Jeśli chcesz, możesz sprawdzić zużycie pamięci przez `%memit`.
W Colab czasem trzeba doinstalować:
```bash
!pip -q install memory_profiler
```
Potem:
- `%load_ext memory_profiler`
- `%memit sum_of_lists(200000)`


## Checklista
- [ ] Umiem użyć `SList.grep` i `SList.fields`.
- [ ] Wiem czemu `!cd` nie działa i używam `%cd`.
- [ ] Umiałem porównać implementacje przez `%timeit -o`.
- [ ] Umiałem użyć `%prun -r` i wskazać bottleneck.
- [ ] Umiałem użyć `%lprun` (line profiler) i na tej podstawie zrefaktorować kod.
